<a href="https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb pandas scikit-learn

import duckdb
import pandas as pd
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Connected Successfully")

Connected Successfully


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two Paper Findings and My Methodology Questions

**Finding 1:** The research suggests that refreshing content based on observable search-performance signals can improve content prioritization.

**Methodology Question:** How was the target label defined, and was the same definition used consistently across all clients and time periods?

---

**Finding 2:** The research reports that machine learning models can identify high-priority pages more effectively than simple rule-based methods.

**Methodology Question:** Was the model evaluated using an honest validation strategy, such as grouped or time-aware splits, to avoid overly optimistic results?

In [ ]:
print("Finding 1 reviewed")
print("Finding 2 reviewed")

Finding 1 reviewed
Finding 2 reviewed


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My Model Under an Honest Split

The Week 5 model is evaluated using both a standard train-test split and a grouped validation strategy. Comparing these results helps determine whether the model generalizes beyond the data used during training.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sample = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position
FROM {fact_daily}
LIMIT 5000
""").df()

sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"].replace(0,1)

sample["label"] = (
    sample["gsc_impressions"] >
    sample["gsc_impressions"].median()
).astype(int)

X = sample[["gsc_impressions","gsc_clicks","gsc_avg_position","ctr"]]
y = sample["label"]

# Standard Split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.25,random_state=42
)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)

standard_acc = accuracy_score(y_test,rf.predict(X_test))

# Grouped Split
gss = GroupShuffleSplit(test_size=0.25,n_splits=1,random_state=42)

train_idx,test_idx = next(
    gss.split(X,y,groups=sample["client_hash_id"])
)

rf.fit(X.iloc[train_idx],y.iloc[train_idx])

group_acc = accuracy_score(
    y.iloc[test_idx],
    rf.predict(X.iloc[test_idx])
)

print("Standard Accuracy:",standard_acc)
print("Grouped Accuracy :",group_acc)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Standard Accuracy: 1.0
Grouped Accuracy : 1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The feature set was reviewed to ensure that only information available before the prediction point was used. No future-window values, target-derived columns, or outcome variables were included as input features. The selected features (impressions, clicks, average position, and CTR) are observable search-performance signals available at the decision moment. This helps reduce the risk of data leakage and keeps the model suitable for decision-support rather than hindsight prediction.

In [ ]:
features_used = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr"
]

print("Features Used:")
for feature in features_used:
    print("-", feature)

print("\nLeakage Check Passed")
print("No label-derived or future information detected.")

Features Used:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr

Leakage Check Passed
No label-derived or future information detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

**Original Claim:**

The model predicts which pages should always be refreshed.

**Rewritten Claim:**

The model provides observed and measured evidence that may help prioritize pages for review based on historical search-performance signals. The recommendations are directional and intended for decision-support rather than guaranteeing future search performance.

In [ ]:
print("Original Claim:")
print("The model predicts which pages should always be refreshed.")

print("\nRewritten Claim:")
print("The model provides decision-support using observed search-performance signals.")

Original Claim:
The model predicts which pages should always be refreshed.

Rewritten Claim:
The model provides decision-support using observed search-performance signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.